# 🤝 Day 01a: TCP — Reliable Data Delivery

---

## 🎯 What You'll Master Today
- TCP 3-way handshake and 4-way termination
- TCP header fields
- Sequence numbers, ACKs, retransmission
- Connection states (LISTEN, ESTABLISHED, TIME_WAIT)

---

```
╔══════════════════════════════════════════════════════════════════════╗
║                TCP 3-WAY HANDSHAKE                                   ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║   Client                              Server                        ║
║     │                                   │                            ║
║     │── SYN (seq=x) ──────────────────→│  1. Client: "Let's talk"  ║
║     │                                   │                            ║
║     │←── SYN-ACK (seq=y, ack=x+1) ────│  2. Server: "OK, ready"   ║
║     │                                   │                            ║
║     │── ACK (ack=y+1) ────────────────→│  3. Client: "Great!"      ║
║     │                                   │                            ║
║     │═══════ CONNECTION ESTABLISHED ════│                            ║
║     │←──────── DATA TRANSFER ──────────→│                            ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. TCP State Machine Simulator
# ============================================

import random

class TCPConnection:
    def __init__(self, role):
        self.role = role  # 'client' or 'server'
        self.state = "CLOSED"
        self.seq = random.randint(1000, 9999)
        self.ack = 0
        self.log = []
    
    def _transition(self, new_state, msg):
        old = self.state
        self.state = new_state
        self.log.append(f"{old} → {new_state}: {msg}")
        print(f"    [{self.role:6}] {old:15} → {new_state:15} | {msg}")

def three_way_handshake():
    print("  ═══ TCP 3-Way Handshake ═══\n")
    
    client = TCPConnection("client")
    server = TCPConnection("server")
    server._transition("LISTEN", "Server starts listening")
    
    # Step 1: SYN
    client._transition("SYN_SENT", f"Send SYN seq={client.seq}")
    
    # Step 2: SYN-ACK
    server.ack = client.seq + 1
    server._transition("SYN_RCVD", f"Send SYN-ACK seq={server.seq} ack={server.ack}")
    
    # Step 3: ACK
    client.ack = server.seq + 1
    client.seq += 1
    client._transition("ESTABLISHED", f"Send ACK ack={client.ack}")
    server._transition("ESTABLISHED", "Connection established")
    
    return client, server

def four_way_termination(client, server):
    print("\n  ═══ TCP 4-Way Termination ═══\n")
    
    # Step 1: FIN
    client._transition("FIN_WAIT_1", "Send FIN")
    
    # Step 2: ACK
    server._transition("CLOSE_WAIT", "Send ACK (received FIN)")
    client._transition("FIN_WAIT_2", "Received ACK")
    
    # Step 3: FIN
    server._transition("LAST_ACK", "Send FIN")
    
    # Step 4: ACK
    client._transition("TIME_WAIT", "Send ACK (wait 2*MSL)")
    server._transition("CLOSED", "Received ACK")
    
    print(f"\n    [client] TIME_WAIT for 2×MSL (typically 60s)")
    client._transition("CLOSED", "Timeout expired")

client, server = three_way_handshake()
four_way_termination(client, server)

In [ ]:
# ============================================
# 2. TCP Header Structure
# ============================================

print("""
  ╔══════════════════════════════════════════════════════════════╗
  ║                TCP HEADER (20 bytes min)                    ║
  ╠══════════════════════════════════════════════════════════════╣
  ║  0                   1                   2         3        ║
  ║  0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5 6 7 8 9 0 1 2 3  ...     ║
  ║  ├─────────────────┼─────────────────┤                     ║
  ║  │  Source Port     │  Dest Port      │  (2+2 = 4 bytes)   ║
  ║  ├─────────────────────────────────────┤                    ║
  ║  │        Sequence Number              │  (4 bytes)         ║
  ║  ├─────────────────────────────────────┤                    ║
  ║  │     Acknowledgment Number           │  (4 bytes)         ║
  ║  ├────┼──────┼─────────────────────────┤                    ║
  ║  │Offs│Reserv│ Flags (URG,ACK,PSH,...) │  (4 bytes)        ║
  ║  ├────┴──────┼─────────────────────────┤                    ║
  ║  │  Window Size    │  Checksum         │  (4 bytes)         ║
  ║  ├─────────────────┼───────────────────┤                    ║
  ║  │  Urgent Pointer │  Options (var)    │  (4+ bytes)        ║
  ║  └─────────────────┴───────────────────┘                    ║
  ╚══════════════════════════════════════════════════════════════╝

  Key flags:
    SYN = Synchronize (start connection)
    ACK = Acknowledge
    FIN = Finish (close connection)
    RST = Reset (abort connection)
    PSH = Push (deliver immediately)
    URG = Urgent (priority data)
""")

In [ ]:
# ============================================
# 3. Sequence Numbers & Retransmission
# ============================================

class TCPReliableTransfer:
    def __init__(self):
        self.seq = 1
        self.acked = {}
    
    def send_data(self, data_chunks, drop_indices=None):
        drop_indices = drop_indices or []
        print("  ═══ TCP Reliable Transfer with Retransmission ═══\n")
        
        for i, chunk in enumerate(data_chunks):
            seq = self.seq
            size = len(chunk)
            
            if i in drop_indices:
                print(f"    → Send seq={seq:4} [{chunk}] ({size}B)  ❌ LOST!")
                print(f"      ... timeout ... retransmitting ...")
                print(f"    → ReTX seq={seq:4} [{chunk}] ({size}B)  ✅")
            else:
                print(f"    → Send seq={seq:4} [{chunk}] ({size}B)  ✅")
            
            self.seq += size
            ack = self.seq
            print(f"    ← ACK  ack={ack:4} (expecting byte {ack} next)")
            print()

transfer = TCPReliableTransfer()
transfer.send_data(
    ["Hello", " Wor", "ld!!"],
    drop_indices=[1]  # Simulate packet 2 being lost
)

print("  Key concepts:")
print("    - seq number = byte offset in stream")
print("    - ACK = next expected byte (cumulative)")
print("    - Timeout → retransmit (RTO = Retransmission Timeout)")
print("    - Fast retransmit: 3 duplicate ACKs → retransmit immediately")

In [ ]:
# ============================================
# 4. TCP Connection States Summary
# ============================================

print("""
  ╔══════════════════════════════════════════════════════════════════╗
  ║               TCP CONNECTION STATES                             ║
  ╠══════════════════════════════════════════════════════════════════╣
  ║  State         │ Side   │ Description                          ║
  ║  ──────────────┼────────┼──────────────────────────────────────║
  ║  CLOSED         │ Both   │ No connection                       ║
  ║  LISTEN         │ Server │ Waiting for SYN                     ║
  ║  SYN_SENT       │ Client │ SYN sent, waiting for SYN-ACK      ║
  ║  SYN_RCVD       │ Server │ SYN received, SYN-ACK sent         ║
  ║  ESTABLISHED    │ Both   │ Connected, data flowing             ║
  ║  FIN_WAIT_1     │ Closer │ FIN sent, waiting for ACK          ║
  ║  FIN_WAIT_2     │ Closer │ ACK received, waiting for peer FIN ║
  ║  CLOSE_WAIT     │ Peer   │ FIN received, ACK sent             ║
  ║  LAST_ACK       │ Peer   │ FIN sent, waiting for final ACK    ║
  ║  TIME_WAIT      │ Closer │ Final ACK sent, waiting 2×MSL      ║
  ╚══════════════════════════════════════════════════════════════════╝

  💼 Why TIME_WAIT? (interview favorite!)
    1. Ensure final ACK reaches the other side
       (if lost, peer retransmits FIN, we can re-ACK)
    2. Prevent old segments from being confused with new connection
       (same src:port + dst:port could be reused)
    Duration: 2 × MSL (Maximum Segment Lifetime) ≈ 60 seconds

  💼 SYN flood attack:
    Attacker sends many SYN packets (spoofed IPs)
    Server fills SYN queue with SYN_RCVD connections
    Legit clients can't connect!
    Fix: SYN cookies (stateless SYN handling)
""")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Explain TCP 3-way handshake | SYN(seq=x) → SYN-ACK(seq=y,ack=x+1) → ACK(ack=y+1). Each side picks ISN |
| 2 | Why is TIME_WAIT needed? | Ensures final ACK arrives + prevents old segments mixing with new connection |
| 3 | What is a SYN flood? | Attacker sends many SYNs, fills server's SYN queue. Fix: SYN cookies |
| 4 | TCP vs UDP? | TCP: reliable, ordered, connection-oriented. UDP: fast, unreliable, connectionless |
| 5 | Fast retransmit? | 3 duplicate ACKs trigger immediate retransmit (don't wait for timeout) |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • 3-way handshake: SYN → SYN-ACK → ACK                │
## │  • 4-way termination: FIN → ACK → FIN → ACK             │
## │  • TIME_WAIT: 2×MSL (~60s) after closing                │
## │  • Seq/ACK numbers track bytes, not packets             │
## │  • Reliability: ACKs + retransmission + checksums       │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **UDP** — Fast, connectionless delivery